# Activations from Scratch

The FFN in a transformer is two linear layers with a single nonlinearity between them — strip the activation out and the whole block collapses to one linear map. This notebook implements every activation covered in the [activations blog post](https://github.com/david-hoangt/david-hoangt.github.io) and runs the small experiments behind the post's claims (dead-ReLU sparsity, GELU≈SiLU overlap, the SwiGLU 8/3 parameter rule):

$$\text{Each activation either reshapes the curve } f(x) \text{ or, for GLU variants, restructures the FFN block itself.}$$

| Section | Activation | Formula | Where it ships |
|---------|-----------|---------|----------------|
| §1 | Sigmoid / Tanh | $1/(1+e^{-x})$, $\tanh x$ | early MLPs / RNN gates |
| §2 | ReLU | $\max(0, x)$ | ResNet, early NLP |
| §3 | GELU | $x\,\Phi(x)$ | BERT, GPT-2/3 |
| §4 | SiLU / Swish | $x\,\sigma(x)$ | gate inside SwiGLU |
| §5 | SwiGLU | $(\mathrm{SiLU}(xW_g)\odot xW_u)\,W_d$ | LLaMA, Mistral, Qwen |
| §6 | ReLU² | $\max(0, x)^2$ | Primer, sparse-MoE research |

Running example: *"The CEO announced record earnings on Friday"* — 7 tokens, `d_model = 64`, `d_ff = 256`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

B, T, d_model, d_ff = 2, 7, 64, 256
tokens = ["The", "CEO", "announced", "record", "earnings", "on", "Friday"]

## §1 — Sigmoid and Tanh: Why They Don't Survive

Both saturate at the tails: once $|x| > 3$ the gradient is essentially zero, so backprop through deep stacks vanishes.

$$\sigma(x) = \frac{1}{1+e^{-x}},\quad \sigma'(x) = \sigma(x)(1-\sigma(x)) \le 0.25 \qquad \tanh'(x) = 1 - \tanh^2 x \le 1$$

| Function | Range | Max gradient | Failure |
|----------|-------|--------------|---------|
| sigmoid | $(0, 1)$ | $0.25$ at $x{=}0$ | vanishes, not zero-centered |
| tanh | $(-1, 1)$ | $1.0$ at $x{=}0$ | vanishes at tails |
| ReLU | $[0, \infty)$ | $1$ for $x>0$ | dead on negatives |

> Plot the functions and their derivatives — watch the sigmoid/tanh slopes flatten past $|x|>3$.

In [ ]:
# ── functions + analytic derivatives on x ∈ [-5, 5] ─────────────────────────
x = torch.linspace(-5, 5, 1000)

sig   = torch.sigmoid(x);            d_sig  = sig * (1 - sig)
tanh  = torch.tanh(x);               d_tanh = 1 - tanh ** 2
relu  = torch.clamp(x, min=0.0);     d_relu = (x > 0).float()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), facecolor="white")
for ax, (s, t, r), ylab in [(ax1, (sig, tanh, relu), "f(x)"),
                            (ax2, (d_sig, d_tanh, d_relu), "f'(x)")]:
    ax.plot(x, s, color="#1E3A8A", lw=2.2, label="sigmoid")
    ax.plot(x, t, color="#3B82F6", lw=2.2, label="tanh")
    ax.plot(x, r, color="#EF4444", lw=2.2, label="ReLU")
    ax.set_xlabel("x"); ax.set_ylabel(ylab)
    ax.grid(True, ls="--", alpha=0.4); ax.legend(loc="upper left")
ax1.set_ylim(-1.2, 5); ax2.set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()

# "CEO" pushed to x=4.2 → sigmoid gradient is already dead
print(f"sigmoid(4.2)  = {torch.sigmoid(torch.tensor(4.2)):.3f}")
print(f"sigmoid'(4.2) = {(torch.sigmoid(torch.tensor(4.2))*(1-torch.sigmoid(torch.tensor(4.2)))):.3f}")

## §2 — ReLU

The standard transformer FFN: project up, apply ReLU, project down. Gradient is exactly 1 on the positive side, 0 on the negative side.

$$\mathrm{FFN}(x) = \mathrm{ReLU}(x W_1 + b_1)\, W_2 + b_2, \qquad \mathrm{ReLU}'(x) = \mathbb{1}[x > 0]$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| $x$ | $(B, T, d_{\text{model}})$ | input, $[2, 7, 64]$ |
| $W_1$ | $(d_{\text{model}}, d_{\text{ff}})$ | up-projection, $[64, 256]$ |
| $W_2$ | $(d_{\text{ff}}, d_{\text{model}})$ | down-projection, $[256, 64]$ |

> ReLU zeros ~half the hidden units on a random forward pass — a sparse representation the next layer routes over.

In [ ]:
class FFN_ReLU(nn.Module):
    """
    Standard transformer FFN with ReLU.

    y = ReLU(x @ W_1 + b_1) @ W_2 + b_2

    Args:
        d_model: Residual-stream width.
        d_ff:    Hidden (intermediate) width, usually 4 * d_model.
    """

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)   # [d_model, d_ff] = [64, 256]
        self.w2 = nn.Linear(d_ff, d_model)   # [d_ff, d_model] = [256, 64]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.w1(x))               # [B, T, d_ff]
        return self.w2(h)                    # [B, T, d_model]

In [ ]:
# ── shape check + measured sparsity of the ReLU hidden layer ────────────────
x = torch.randn(B, T, d_model)
ffn = FFN_ReLU(d_model, d_ff)
out = ffn(x)

h = F.relu(ffn.w1(x))                         # [B, T, d_ff]
dead_frac = (h == 0).float().mean().item()

print(f"Output shape: {tuple(out.shape)}")    # (2, 7, 64)
print(f"Zeroed activations: {dead_frac:.1%} of {B*T*d_ff} entries")
assert out.shape == (B, T, d_model)
assert 0.3 < dead_frac < 0.7                  # roughly half fire

## §3 — GELU

ReLU multiplies $x$ by a hard 0/1 mask; GELU weights $x$ by the probability a standard normal sits below it, giving a smooth dip below zero (min $\approx -0.17$ at $x \approx -0.75$).

$$\mathrm{GELU}(x) = x\,\Phi(x),\quad \Phi(x) = \tfrac{1}{2}\!\left(1 + \mathrm{erf}\tfrac{x}{\sqrt 2}\right) \qquad \approx 0.5x\left(1 + \tanh\!\big[\sqrt{2/\pi}\,(x + 0.044715 x^3)\big]\right)$$

| Variant | PyTorch call | Notes |
|---------|-------------|-------|
| exact | `F.gelu(x)` | erf-based, default |
| tanh approx | `F.gelu(x, approximate="tanh")` | HF `gelu_new`, faster |

> The exact and tanh forms differ by ~$10^{-3}$ — match the model card when reproducing a checkpoint.

In [ ]:
class FFN_GELU(nn.Module):
    """
    BERT/GPT-2 style FFN — same structure as the ReLU FFN, GELU swapped in.

    Args:
        d_model: Residual-stream width.
        d_ff:    Hidden width.
    """

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.gelu(self.w1(x))               # exact erf-based by default
        return self.w2(h)                    # [B, T, d_model]

In [ ]:
# ── exact vs tanh approximation: numerically close, not identical ───────────
x = torch.linspace(-4, 4, 1000)
exact = F.gelu(x)
approx = F.gelu(x, approximate="tanh")
print(f"max |exact - tanh approx| = {(exact - approx).abs().max():.2e}")

# GELU's negative lobe: minimum value and its location
xmin = x[exact.argmin()]
print(f"GELU min = {exact.min():.3f} at x = {xmin:.3f}")
assert (exact - approx).abs().max() < 2e-3

## §4 — SiLU / Swish

The cheaper GELU twin: one sigmoid, one multiply, no `erf`. As a standalone activation it never beat GELU by enough to matter — its lasting role is the gate inside SwiGLU.

$$\mathrm{SiLU}(x) = x\,\sigma(x) = \frac{x}{1 + e^{-x}}$$

| | SiLU | GELU |
|---|------|------|
| cost | sigmoid + mul | erf (or tanh+cube) |
| min | $\approx -0.28$ at $x{\approx}-1.28$ | $\approx -0.17$ at $x{\approx}-0.75$ |

> SiLU and GELU overlap so closely you have to plot the difference to tell them apart.

In [ ]:
class FFN_SiLU(nn.Module):
    """
    Drop-in for the ReLU/GELU FFN — SiLU (x * sigmoid(x)) swapped in.

    Args:
        d_model: Residual-stream width.
        d_ff:    Hidden width.
    """

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.silu(self.w1(x))               # x * sigmoid(x)
        return self.w2(h)                    # [B, T, d_model]

In [ ]:
# ── how close are SiLU and GELU? plot both + their difference ───────────────
x = torch.linspace(-4, 4, 1000)
silu, gelu = F.silu(x), F.gelu(x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), facecolor="white")
ax1.plot(x, gelu, color="#3B82F6", lw=2.2, label="GELU")
ax1.plot(x, silu, color="#EF4444", lw=2.2, ls="--", label="SiLU")
ax1.axhline(0, color="k", lw=.8); ax1.legend(); ax1.grid(True, ls="--", alpha=.4)
ax1.set_title("GELU vs SiLU"); ax1.set_xlabel("x"); ax1.set_ylabel("f(x)")
ax2.plot(x, gelu - silu, color="#10B981", lw=2.2)
ax2.set_title("GELU − SiLU"); ax2.set_xlabel("x"); ax2.grid(True, ls="--", alpha=.4)
plt.tight_layout(); plt.show()

print(f"max |GELU - SiLU| = {(gelu - silu).abs().max():.3f}")

## §5 — SwiGLU and the 8/3 Rule

GLU splits the up-projection in two: one path is gated by SiLU, then multiplied into the other. Three matrices instead of two — so shrink `d_ff` by $8/3$ to keep the parameter count of a $4\times$ FFN.

$$\mathrm{SwiGLU}(x) = \big(\mathrm{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}})\big)\, W_{\text{down}}, \qquad d_{\text{ff}}^{\text{SwiGLU}} = \tfrac{8}{3}\, d_{\text{model}}$$

| Matrix | Shape | Path |
|--------|-------|------|
| $W_{\text{gate}}$ | $(d_{\text{model}}, d_{\text{ff}})$ | gate → SiLU |
| $W_{\text{up}}$ | $(d_{\text{model}}, d_{\text{ff}})$ | value, no activation |
| $W_{\text{down}}$ | $(d_{\text{ff}}, d_{\text{model}})$ | back to residual |

> No biases — modern LLMs drop them, same convention as Q/K/V.

In [ ]:
class SwiGLU_FFN(nn.Module):
    """
    LLaMA-style gated FFN.

    y = ( SiLU(x @ W_gate) ⊙ (x @ W_up) ) @ W_down

    Three projections, no biases. Use d_ff ≈ (8/3) * d_model to match the
    parameter budget of a 4×-expanded two-matrix FFN.

    Args:
        d_model: Residual-stream width.
        d_ff:    Gated hidden width (≈ 8/3 * d_model).
    """

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.w_gate(x))        # [B, T, d_ff]
        up   = self.w_up(x)                  # [B, T, d_ff]
        return self.w_down(gate * up)        # [B, T, d_model]

In [ ]:
# ── the 8/3 rule: SwiGLU with shrunk d_ff ≈ params of a 4× ReLU FFN ─────────
d_ff_relu   = 4 * d_model                                   # 256
d_ff_swiglu = round(8 / 3 * d_model / 8) * 8                # 168, tile-friendly

relu_ffn   = FFN_ReLU(d_model, d_ff_relu)
swiglu_ffn = SwiGLU_FFN(d_model, d_ff_swiglu)

def n_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

x = torch.randn(B, T, d_model)
print(f"ReLU   FFN  d_ff={d_ff_relu:<4} params={n_params(relu_ffn):,}")
print(f"SwiGLU FFN  d_ff={d_ff_swiglu:<4} params={n_params(swiglu_ffn):,}")
print(f"SwiGLU output shape: {tuple(swiglu_ffn(x).shape)}")   # (2, 7, 64)

# weight-only counts (drop ReLU's biases) land within ~5%
w_relu   = 2 * d_model * d_ff_relu
w_swiglu = 3 * d_model * d_ff_swiglu
print(f"weight params  ReLU={w_relu:,}  SwiGLU={w_swiglu:,}  ratio={w_swiglu/w_relu:.2f}")
assert swiglu_ffn(x).shape == (B, T, d_model)

## §6 — ReLU² and the Quiet Revival

Squared ReLU: same zero floor, quadratic growth on the positive side. Primer (So et al., 2021) found it competitive on autoregressive LMs; it resurfaces in some sparse-MoE work.

$$\mathrm{ReLU}^2(x) = \max(0, x)^2$$

> Cheap (one max, one multiply, no exponentials) and sparse — but it did not displace SwiGLU as the default.

In [ ]:
def relu_squared(x: torch.Tensor) -> torch.Tensor:
    """Squared ReLU: max(0, x)**2 — zero on negatives, quadratic on positives."""
    return F.relu(x) ** 2


class FFN_ReLU2(nn.Module):
    """FFN with squared-ReLU activation (Primer, 2021)."""

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(relu_squared(self.w1(x)))   # [B, T, d_model]


x = torch.randn(B, T, d_model)
print(f"ReLU2 FFN output shape: {tuple(FFN_ReLU2(d_model, d_ff)(x).shape)}")

## §7 — All Activations on One Axis

The trajectory: saturating S-curves → hard ReLU → smooth GELU/SiLU → quadratic ReLU². (SwiGLU is structural, not a single curve, so it is not plotted here.)

In [ ]:
# ── overlay every point-wise activation on x ∈ [-4, 4] ──────────────────────
x = torch.linspace(-4, 4, 1000)
curves = {
    "ReLU":   (F.relu(x),        "#1E3A8A"),
    "GELU":   (F.gelu(x),        "#3B82F6"),
    "SiLU":   (F.silu(x),        "#EF4444"),
    "ReLU²":  (relu_squared(x),  "#10B981"),
    "tanh":   (torch.tanh(x),    "#F59E0B"),
}
plt.figure(figsize=(9, 6), facecolor="white")
for name, (y, c) in curves.items():
    plt.plot(x, y, color=c, lw=2.2, label=name)
plt.axhline(0, color="k", lw=.8); plt.axvline(0, color="grey", ls="--", lw=1)
plt.ylim(-1.2, 4); plt.xlabel("x"); plt.ylabel("f(x)")
plt.grid(True, ls="--", alpha=.4); plt.legend(loc="upper left")
plt.title("Point-wise activations"); plt.tight_layout(); plt.show()